# 01 — Ship Generation

Generates synthetic ships from bilateral trade weight matrices.

**Prerequisites:**
1. Run `simulation_config.ipynb` to produce `simulation_config.json`.
2. Run `00_precompute_routes.ipynb` to produce `port_pair_routes.pkl` and `country_pair_optimal.pkl`.

**Output:** `simulation_output_data/ships.parquet`  
Also stores `common_countries.json`, `country_to_ports.json`, and `port_name_to_node.pkl` for use by `02_simulation.ipynb`.

---

In [1]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm

# Add simulation_engine to path
sys.path.insert(0, str(Path('.').resolve()))

from simulation_engine.config_loader import load_config, resolve_paths, get_economic_events
from simulation_engine.event_manager import build_epoch_schedule
from simulation_engine.ship_generation import generate_all_ships
from simulation_engine.io_manager import write_parquet

print('Imports loaded successfully.')

Imports loaded successfully.


In [2]:
# ============================================================
# Load configuration
# ============================================================
cfg = load_config()                  # reads simulation_config.json
cfg = resolve_paths(cfg)             # resolve relative paths to absolute

print('Configuration loaded.')
print(f'  Simulation: {cfg["SIMULATION_DAYS"]} days  |  interval: {cfg["INTERVAL_SIZE"]*24:.1f} h')
print(f'  HS codes: {len(cfg["HS_CODES_LIST"])}')
print(f'  Economic events: {len(cfg["ECONOMIC_EVENTS"])}')
print(f'  Interruption events: {len(cfg["INTERRUPTION_EVENTS"])}')
print(f'  Port weight size/type: {cfg["PORT_WEIGHT_SIZE"]}/{cfg["PORT_WEIGHT_TYPE"]}')
print(f'  Distance penalty scale: {cfg["DISTANCE_PENALTY_SCALE"]}')

Configuration loaded.
  Simulation: 365 days  |  interval: 1.0 h
  HS codes: 96
  Economic events: 0
  Interruption events: 1
  Port weight size/type: 0.5/0.5
  Distance penalty scale: 1


In [3]:
# ============================================================
# Load network and pre-computed routes
# ============================================================
output_dir = Path(cfg['OUTPUT_DIR'])

print('Loading network...')
with open(cfg['NETWORK_FILE'], 'rb') as f:
    G = pickle.load(f)

port_count  = sum(1 for n in G.nodes() if G.nodes[n].get('source') == 'port')
choke_count = sum(1 for n in G.nodes() if G.nodes[n].get('source') == 'choke_point')
print(f'  Nodes: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}')
print(f'  Ports: {port_count}  |  Choke points: {choke_count}')

# Load pre-computed port-pair routes (from 00_precompute_routes.ipynb)
port_pair_routes_path = output_dir / 'port_pair_routes.pkl'
country_optimal_path  = output_dir / 'country_pair_optimal.pkl'

if not port_pair_routes_path.exists():
    raise FileNotFoundError(
        f"'{port_pair_routes_path}' not found.\n"
        "Run 00_precompute_routes.ipynb first."
    )
if not country_optimal_path.exists():
    raise FileNotFoundError(
        f"'{country_optimal_path}' not found.\n"
        "Run 00_precompute_routes.ipynb first."
    )

print('\nLoading pre-computed routes...')
with open(port_pair_routes_path, 'rb') as f:
    port_pair_routes = pickle.load(f)
with open(country_optimal_path, 'rb') as f:
    country_pair_optimal = pickle.load(f)

print(f'  Port-pair routes:       {len(port_pair_routes):,}')
print(f'  Country-pair optimal:   {len(country_pair_optimal):,}')

Loading network...
  Nodes: 8726  |  Edges: 14634
  Ports: 595  |  Choke points: 28

Loading pre-computed routes...


  Port-pair routes:       345,156
  Country-pair optimal:   28,056


In [4]:
# ============================================================
# Random number generator
# ============================================================
seed = cfg.get('RANDOM_SEED')
rng  = np.random.default_rng(seed)
print(f'RNG initialised with seed={seed}')

RNG initialised with seed=42


In [5]:
# ============================================================
# Load IMF port data (for port selection weights)
# ============================================================
# The IMF port data provides import/export shares and vessel-type counts
# used to weight port selection within each country.
# baci_name must be present — it maps ISO3 codes to BACI trade-data country names.

imf_port_path = cfg['IMF_PORT_DATA_FILE']
baci_codes_path = cfg['BACI_CODES_FILE']

imf_port_df = pd.read_csv(imf_port_path)

# Build ISO3 → BACI country name mapping (same logic as network_extraction.ipynb)
baci_codes   = pd.read_csv(baci_codes_path)
iso3_to_baci = dict(zip(baci_codes['country_iso3'], baci_codes['country_name']))
iso3_to_baci['TWN'] = iso3_to_baci['S19']   # Taiwan → 'Other Asia, nes'
imf_port_df['baci_name'] = imf_port_df['ISO3'].map(iso3_to_baci)

# Filter to ports with a BACI country mapping (same as network_extraction)
imf_port_df = imf_port_df[imf_port_df['baci_name'].notna()].copy()

print(f'IMF port data loaded: {len(imf_port_df):,} ports across {imf_port_df["ISO3"].nunique()} countries')
imf_port_df[['portname', 'baci_name', 'share_country_maritime_export',
             'share_country_maritime_import', 'vessel_count_tanker',
             'vessel_count_dry_bulk', 'vessel_count_container']].head()

IMF port data loaded: 2,017 ports across 172 countries


,portname,baci_name,share_country_maritime_export,share_country_maritime_import,vessel_count_tanker,vessel_count_dry_bulk,vessel_count_container
0,Tsuruga,Japan,0.08,0.42,54,96,77
1,Sibolga,Indonesia,0.01,0.07,39,2,28
2,Fangcheng,China,0.42,1.89,210,1360,34
3,Ube,Japan,2.35,0.75,2088,1195,13
4,Bluff Harbor,New Zealand,2.84,7.78,58,112,43


In [6]:
# ============================================================
# Build epoch schedule
# ============================================================
all_economic_events = get_economic_events(cfg)
baseline_events = [e for e in all_economic_events if e.day == 0]
mid_sim_events  = [e for e in all_economic_events if e.day > 0]

epoch_schedule = build_epoch_schedule(
    simulation_days=cfg['SIMULATION_DAYS'],
    economic_events=mid_sim_events,
)

print(f'Epoch schedule: {len(epoch_schedule)} epoch(s)')
for ep in epoch_schedule:
    n_adj = len(ep['cumulative_adjustments'])
    print(f"  Day {ep['start_day']:.0f} – {ep['end_day']:.0f}  ({n_adj} cumulative adjustment(s))")

Epoch schedule: 1 epoch(s)
  Day 0 – 365  (0 cumulative adjustment(s))


In [7]:
# ============================================================
# Generate ships
# ============================================================
output_dir.mkdir(parents=True, exist_ok=True)

print('\nGenerating ships...')

all_ships, common_countries, country_to_ports, port_name_to_node = generate_all_ships(
    cfg=cfg,
    G=G,
    port_pair_routes=port_pair_routes,
    country_pair_optimal=country_pair_optimal,
    imf_port_df=imf_port_df,
    economic_events_baseline=baseline_events,
    epoch_schedule=epoch_schedule,
    rng=rng,
    show_progress=True,
)

print(f'\nGenerated {len(all_ships):,} ships')
print(f'Countries in simulation: {len(common_countries)}')
print(f'  {common_countries}')


Generating ships...


Building port selection weights from IMF data...
  Port selection data built for 168 countries.


Epochs:   0%|          | 0/1 [00:00<?, ?epoch/s]

Generating ships (day 0–365):   0%|          | 0/225124 [00:00<?, ?it/s]

Generating ships (day 0–365):   0%|          | 635/225124 [00:00<00:35, 6342.64it/s]

...

Generating ships (day 0–365): 100%|█████████▉| 224600/225124 [00:22<00:00, 12920.34it/s]

Generating ships (day 0–365): 100%|██████████| 225124/225124 [00:22<00:00, 10009.38it/s]


Epochs: 100%|██████████| 1/1 [00:54<00:00, 54.42s/epoch]

Epochs: 100%|██████████| 1/1 [00:54<00:00, 54.42s/epoch]


Generated 225,124 ships
Countries in simulation: 168
  ['Albania', 'Algeria', 'American Samoa', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Argentina', 'Aruba', 'Australia', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belgium', 'Belize', 'Benin', 'Bonaire', 'Br. Virgin Isds', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Cayman Isds', 'Chile', 'China', 'China, Hong Kong SAR', 'China, Macao SAR', 'Colombia', 'Comoros', 'Congo', 'Cook Isds', 'Costa Rica', 'Croatia', 'Cuba', 'CuraÃ§ao', 'Cyprus', "CÃ´te d'Ivoire", 'Dem. Rep. of the Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Rep.', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'FS Micronesia', 'Fiji', 'Finland', 'France', 'French Polynesia', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Gibraltar', 'Greece', 'Grenada', 'Guam', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Honduras', 'Iceland', 'India', 'Indonesia', 'Iran', 'I

In [8]:
# ============================================================
# Summary statistics
# ============================================================
from collections import Counter

type_counts = Counter(s.ship_type for s in all_ships)
total_weight = sum(s.cargo_total_weight for s in all_ships)
total_value  = sum(s.cargo_total_value  for s in all_ships)

print('=' * 60)
print('SHIP GENERATION SUMMARY')
print('=' * 60)
print(f'Total ships:  {len(all_ships):,}')
print(f'Total weight: {total_weight:,.0f} metric tons')
print(f'Total value:  ${total_value:,.0f}')
print()
print('Ship type breakdown:')
for st, cnt in type_counts.most_common():
    pct = cnt / len(all_ships) * 100
    print(f'  {st.title():<15} {cnt:6,}  ({pct:.1f}%)')
print('=' * 60)

SHIP GENERATION SUMMARY
Total ships:  225,124
Total weight: 13,376,931,659 metric tons
Total value:  $19,171,182,821,102

Ship type breakdown:
  Tanker          115,876  (51.5%)
  Bulk Carrier    71,839  (31.9%)
  Cargo Ship      37,409  (16.6%)


In [9]:
# ============================================================
# Export ships.parquet
# ============================================================
records = [s.to_record() for s in all_ships]
ships_df = pd.DataFrame(records)

ships_parquet = output_dir / 'ships.parquet'
write_parquet(ships_df, str(ships_parquet), append=False)
print(f'Exported ships.parquet  ({len(ships_df):,} rows, {len(ships_df.columns)} columns)')
print(f'  Size: {ships_parquet.stat().st_size / 1024:.0f} KB')

Exported ships.parquet  (225,124 rows, 204 columns)
  Size: 51623 KB


In [10]:
# ============================================================
# Export auxiliary files for 02_simulation.ipynb
# ============================================================

# common_countries.json
with open(output_dir / 'common_countries.json', 'w') as f:
    json.dump(common_countries, f)
print('Exported common_countries.json')

# country_to_ports.json
with open(output_dir / 'country_to_ports.json', 'w') as f:
    json.dump(country_to_ports, f)
print('Exported country_to_ports.json')

# port_name_to_node.pkl (node IDs may not be JSON-serialisable)
with open(output_dir / 'port_name_to_node.pkl', 'wb') as f:
    pickle.dump(port_name_to_node, f)
print('Exported port_name_to_node.pkl')

print()
print('Next step: run 02_simulation.ipynb')

Exported common_countries.json
Exported country_to_ports.json
Exported port_name_to_node.pkl

Next step: run 02_simulation.ipynb
